In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time

class HandTracker:
    def __init__(self, confidence=0.8):
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(min_detection_confidence=confidence)

    def find_landmarks(self, img_rgb):
        results = self.hands.process(img_rgb)
        if results.multi_hand_landmarks:
            return results.multi_hand_landmarks[0].landmark
        return None

class UIHandler:
    def __init__(self, click_delay=0.4):
        self.mp_selfie = mp.solutions.selfie_segmentation
        self.selfie = self.mp_selfie.SelfieSegmentation(model_selection=1)
        self.click_delay = click_delay
        self.hover_start_time = 0
        self.selected_box = None
        
        # [x1, y1, x2, y2, name, color]
        self.buttons = [
            [50, 10, 200, 80, "pencil", (0, 255, 255)],
            [220, 10, 370, 80, "eraser", (0, 255, 255)],
            [390, 10, 540, 80, "clear", (0, 255, 255)]
        ]

    def apply_blur(self, img, img_rgb):
        results = self.selfie.process(img_rgb)
        mask = results.segmentation_mask > 0.1
        mask_3d = np.dstack((mask, mask, mask))
        blurred_img = cv2.GaussianBlur(img, (55, 55), 0)
        return np.where(mask_3d, img, blurred_img)

    def draw_buttons(self, img, current_tool):
        for x1, y1, x2, y2, name, color in self.buttons:
            thick = -1 if current_tool == name else 2
            cv2.rectangle(img, (x1, y1), (x2, y2), color, thick)
            text_color = (0, 0, 0) if current_tool == name else (0, 255, 255)
            cv2.putText(img, name.upper(), (x1+20, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2)
        return img

    def check_button_click(self, img, cx, cy):
        clicked_tool = None
        in_button = False
        
        for x1, y1, x2, y2, name, color in self.buttons:
            if x1 < cx < x2 and y1 < cy < y2:
                in_button = True
                if self.selected_box != name:
                    self.selected_box = name
                    self.hover_start_time = time.time()

                elapsed = time.time() - self.hover_start_time
                progress = min(elapsed / self.click_delay, 1.0)
                
                # Feedback visuals
                cv2.circle(img, (cx, cy), 15, (0, 255, 0), 2)
                cv2.ellipse(img, (cx, cy), (15, 15), 0, 0, progress*360, (255, 255, 255), 3)

                if elapsed >= self.click_delay:
                    clicked_tool = name
                break
        
        if not in_button:
            self.selected_box = None
            
        return clicked_tool

class AirCanvas:
    def __init__(self):
        self.cap = cv2.VideoCapture(0)
        self.tracker = HandTracker()
        self.ui = UIHandler()
        self.canvas = None
        self.current_tool = "pencil"
        self.prev_x, self.prev_y = 0, 0

    def run(self):
        while self.cap.isOpened():
            success, img = self.cap.read()
            if not success: break

            img = cv2.flip(img, 1)
            h, w, _ = img.shape
            if self.canvas is None: self.canvas = np.zeros_like(img)

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # 1. Background Blur
            img = self.ui.apply_blur(img, img_rgb)
            
            # 2. UI Rendering
            img = self.ui.draw_buttons(img, self.current_tool)

            # 3. Process Hand Landmarks
            landmarks = self.tracker.find_landmarks(img_rgb)
            if landmarks:
                cx, cy = int(landmarks[8].x * w), int(landmarks[8].y * h)
                index_up = landmarks[8].y < landmarks[6].y

                # Check for clicks
                new_tool = self.ui.check_button_click(img, cx, cy)
                if new_tool:
                    if new_tool == "clear":
                        self.canvas = np.zeros_like(img)
                    else:
                        self.current_tool = new_tool

                # Drawing logic
                if index_up and cy > 100:
                    if self.prev_x == 0: self.prev_x, self.prev_y = cx, cy
                    
                    if self.current_tool == "pencil":
                        cv2.line(self.canvas, (self.prev_x, self.prev_y), (cx, cy), (0, 255, 255), 5)
                    elif self.current_tool == "eraser":
                        cv2.line(self.canvas, (self.prev_x, self.prev_y), (cx, cy), (0, 0, 0), 50)
                    
                    self.prev_x, self.prev_y = cx, cy
                else:
                    self.prev_x, self.prev_y = 0, 0

                cv2.circle(img, (cx, cy), 5, (255, 255, 255), -1)

            # 4. Merge Canvas
            img = self._merge_canvas(img)

            cv2.imshow("OOP Air Canvas", img)
            if cv2.waitKey(1) & 0xFF == 27: break

        self.cap.release()
        cv2.destroyAllWindows()

    def _merge_canvas(self, img):
        canvas_gray = cv2.cvtColor(self.canvas, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(canvas_gray, 1, 255, cv2.THRESH_BINARY)
        img[mask > 0] = self.canvas[mask > 0]
        return img

if __name__ == "__main__":
    app = AirCanvas()
    app.run()